
# Genre Popularity Trends
## Performance Benchmarking Implementation

### Analysis Overview:
- **Q1: Explode multi-genre strings**: Converts semi-structured genre data into analyzable rows; causes dataset expansion
  *Metric: Explosion time + Peak memory usage*

- **Q2: Yearly average rating by genre**: Core trend metric; classic MapReduce pattern (GroupBy + aggregation)
  *Metric: GroupBy time + Rows/sec*

- **Q3: Genre volume over time**: Tracks production trends per genre
  *Metric: Count aggregation speed + Memory*

- **Q4: Weighted popularity score**: Improves metric quality using vote_count weighting
  *Metric: CPU time + Column memory cost*

- **Q5: Top performing genre per year**: Requires ranking across groups
  *Metric: Ranking time + Sorting overhead*


In [2]:
import pandas as pd
import numpy as np
import time
import psutil
import os
import ast
import sys
from datetime import datetime


In [3]:
print("\nSTARTING ANALYSIS")
print("-"*40)

start_total = time.time()
start_memory_total = psutil.Process(os.getpid()).memory_info().rss / (1024 * 1024)
start_cpu_total = psutil.Process(os.getpid()).cpu_percent(interval=None)

print(f"Initial memory: {start_memory_total:.2f} MB")


STARTING ANALYSIS
----------------------------------------
Initial memory: 136.31 MB


In [4]:
print("\nLOADING DATA")
print("-"*40)

start_time = time.time()
start_memory = psutil.Process(os.getpid()).memory_info().rss / (1024 * 1024)

movies = pd.read_csv("movies_cleaned.csv")

end_time = time.time()
end_memory = psutil.Process(os.getpid()).memory_info().rss / (1024 * 1024)

execution_time = end_time - start_time
memory_delta = end_memory - start_memory

# Store metrics for summary
load_time = execution_time
load_memory = memory_delta
load_end_memory = end_memory

print(f"\n BENCHMARKS:")
print(f"Time: {execution_time:.2f} seconds")
print(f"Memory delta: {memory_delta:.2f} MB")
print(f"Memory total: {end_memory:.2f} MB")
print(f"Rows loaded: {len(movies):,}")
print(f"Columns: {len(movies.columns)}")
print(f"Initial Memory (MB): {round(movies.memory_usage(deep=True).sum() / 1024**2, 2):,}")


LOADING DATA
----------------------------------------

 BENCHMARKS:
Time: 9.13 seconds
Memory delta: 2254.89 MB
Memory total: 2391.34 MB
Rows loaded: 590,202
Columns: 166
Initial Memory (MB): 2,135.94


In [5]:
# DATA PREPARATION
print("\nPREPARING DATA")
print("-"*40)

start_time = time.time()
start_memory = psutil.Process(os.getpid()).memory_info().rss / (1024 * 1024)

# Ensure release_date is datetime
movies["release_date"] = pd.to_datetime(movies["release_date"], errors="coerce")

# Create year column
movies["year"] = movies["release_date"].dt.year

# Ensure genres is list
def parse_list(x):
    if isinstance(x, str):
        try:
            if x.startswith("["):
                return ast.literal_eval(x)
            else:
                return [i.strip() for i in x.split(",")]
        except:
            return ["Unknown"]
    return x

movies["genres"] = movies["genres"].apply(parse_list)

# Store preparation metrics
prep_time = time.time() - start_time
prep_memory = psutil.Process(os.getpid()).memory_info().rss / (1024 * 1024) - start_memory

print(f"\n BENCHMARKS:")
print(f"Time: {prep_time:.2f} seconds")
print(f"Memory delta: {prep_memory:.2f} MB")
print(f"Memory total: {psutil.Process(os.getpid()).memory_info().rss / (1024 * 1024):.2f} MB")


PREPARING DATA
----------------------------------------

 BENCHMARKS:
Time: 0.40 seconds
Memory delta: 129.52 MB
Memory total: 2534.44 MB


In [6]:
print("\n" + "="*80)
print("Q1: EXPLODE MULTI-GENRE STRINGS")
print("="*80)

start_time = time.time()
start_memory = psutil.Process(os.getpid()).memory_info().rss / (1024 * 1024)
start_cpu = psutil.Process(os.getpid()).cpu_percent(interval=None)

# Explode genres
df_exploded = movies.explode("genres")

end_time = time.time()
end_memory = psutil.Process(os.getpid()).memory_info().rss / (1024 * 1024)
end_cpu = psutil.Process(os.getpid()).cpu_percent(interval=None)

execution_time = end_time - start_time
memory_delta = end_memory - start_memory
rows_after_explosion = len(df_exploded)
expansion_factor = rows_after_explosion / len(movies)

# Store metrics for summary
analysis1_time = execution_time
analysis1_memory = memory_delta

print(f"\n RESULTS:")
print(f"Rows before explosion: {len(movies):,}")
print(f"Rows after explosion: {rows_after_explosion:,}")
print(f"Expansion factor: {expansion_factor:.2f}x")
print(f"New rows added: {rows_after_explosion - len(movies):,}")

print(f"\n BENCHMARKS:")
print(f"Time: {execution_time:.2f} seconds")
print(f"Memory delta: {memory_delta:.2f} MB")
print(f"Memory total: {end_memory:.2f} MB")
print(f"CPU: {end_cpu:.1f}%")
print(f"Explosion speed: {rows_after_explosion/execution_time:,.0f} rows/second")


Q1: EXPLODE MULTI-GENRE STRINGS

 RESULTS:
Rows before explosion: 590,202
Rows after explosion: 976,441
Expansion factor: 1.65x
New rows added: 386,239

 BENCHMARKS:
Time: 1.66 seconds
Memory delta: 783.77 MB
Memory total: 3318.39 MB
CPU: 0.0%
Explosion speed: 589,980 rows/second


In [7]:
print("\n" + "="*80)
print("Q2: YEARLY AVERAGE RATING BY GENRE")
print("="*80)

start_time = time.time()
start_memory = psutil.Process(os.getpid()).memory_info().rss / (1024 * 1024)
start_cpu = psutil.Process(os.getpid()).cpu_percent(interval=None)

# GroupBy aggregation for average rating
grouped = df_exploded.groupby(["year", "genres"])["vote_average"].mean().round(2).reset_index()

end_time = time.time()
end_memory = psutil.Process(os.getpid()).memory_info().rss / (1024 * 1024)
end_cpu = psutil.Process(os.getpid()).cpu_percent(interval=None)

execution_time = end_time - start_time
memory_delta = end_memory - start_memory
rows_processed = len(df_exploded)
rows_per_sec = rows_processed / execution_time if execution_time > 0 else 0
result_rows = len(grouped)

# Store metrics for summary
analysis2_time = execution_time
analysis2_memory = memory_delta

print(f"\n RESULTS:")
print(f"Unique year-genre combinations: {result_rows:,}")
print(f"Unique years: {grouped['year'].nunique()}")
print(f"Unique genres: {grouped['genres'].nunique()}")
print("\nSample results (first 10 rows):")
print(grouped.head(10))

print(f"\n BENCHMARKS:")
print(f"Time: {execution_time:.2f} seconds")
print(f"Memory delta: {memory_delta:.2f} MB")
print(f"Memory total: {end_memory:.2f} MB")
print(f"CPU: {end_cpu:.1f}%")
print(f"Rows processed: {rows_processed:,}")
print(f"Rows/second: {rows_per_sec:,.0f}")
print(f"GroupBy reduction ratio: {rows_processed/result_rows:.1f}:1")


Q2: YEARLY AVERAGE RATING BY GENRE

 RESULTS:
Unique year-genre combinations: 2,433
Unique years: 143
Unique genres: 19

Sample results (first 10 rows):
   year       genres  vote_average
0  1888  Documentary          5.68
1  1890  Documentary          4.70
2  1891  Documentary          4.90
3  1892  Documentary          3.50
4  1893  Documentary          1.85
5  1893        Drama          4.15
6  1894       Action          5.20
7  1894       Comedy          5.15
8  1894        Crime          5.05
9  1894  Documentary          4.37

 BENCHMARKS:
Time: 0.06 seconds
Memory delta: 31.88 MB
Memory total: 3350.44 MB
CPU: 0.0%
Rows processed: 976,441
Rows/second: 16,229,920
GroupBy reduction ratio: 401.3:1


In [8]:
print("\n" + "="*80)
print("Q3: GENRE VOLUME OVER TIME")
print("="*80)

start_time = time.time()
start_memory = psutil.Process(os.getpid()).memory_info().rss / (1024 * 1024)
start_cpu = psutil.Process(os.getpid()).cpu_percent(interval=None)

# Count aggregation for genre volume
volume = df_exploded.groupby(["year", "genres"]).size().reset_index(name="movie_count")

end_time = time.time()
end_memory = psutil.Process(os.getpid()).memory_info().rss / (1024 * 1024)
end_cpu = psutil.Process(os.getpid()).cpu_percent(interval=None)

execution_time = end_time - start_time
memory_delta = end_memory - start_memory
rows_processed = len(df_exploded)
rows_per_sec = rows_processed / execution_time if execution_time > 0 else 0
result_rows = len(volume)

# Store metrics for summary
analysis3_time = execution_time
analysis3_memory = memory_delta

print(f"\n RESULTS:")
print(f"Total year-genre volume records: {result_rows:,}")
print(f"Total movies counted: {volume['movie_count'].sum():,}")
print(f"Average movies per year-genre: {volume['movie_count'].mean():.1f}")
print(f"Max movies in a single year-genre: {volume['movie_count'].max()}")
print("\nTop 10 highest volume year-genre combinations:")
print(volume.sort_values("movie_count", ascending=False).head(10))

print(f"\n BENCHMARKS:")
print(f"Time: {execution_time:.2f} seconds")
print(f"Memory delta: {memory_delta:.2f} MB")
print(f"Memory total: {end_memory:.2f} MB")
print(f"CPU: {end_cpu:.1f}%")
print(f"Rows processed: {rows_processed:,}")
print(f"Rows/second: {rows_per_sec:,.0f}")
print(f"Count aggregation speed: {result_rows/execution_time:,.0f} groups/second")


Q3: GENRE VOLUME OVER TIME

 RESULTS:
Total year-genre volume records: 2,433
Total movies counted: 976,441
Average movies per year-genre: 401.3
Max movies in a single year-genre: 13518

Top 10 highest volume year-genre combinations:
      year  genres  movie_count
2354  2025   Drama        13518
2335  2024   Drama        13456
2316  2023   Drama        11826
2297  2022   Drama        10006
2278  2021   Drama         8674
2240  2019   Drama         8372
2221  2018   Drama         7767
2332  2024  Comedy         7431
2259  2020   Drama         7400
2351  2025  Comedy         7378

 BENCHMARKS:
Time: 0.05 seconds
Memory delta: 23.89 MB
Memory total: 3374.73 MB
CPU: 0.0%
Rows processed: 976,441
Rows/second: 18,130,303
Count aggregation speed: 45,175 groups/second


In [9]:
print("\n" + "="*80)
print("Q4: WEIGHTED POPULARITY SCORE")
print("="*80)

start_time = time.time()
start_memory = psutil.Process(os.getpid()).memory_info().rss / (1024 * 1024)
start_cpu = psutil.Process(os.getpid()).cpu_percent(interval=None)

# Calculate weighted popularity score (IMDb formula)
C = df_exploded["vote_average"].mean()
m = df_exploded["vote_count"].quantile(0.75)

df_exploded["weighted_score"] = (
    (df_exploded["vote_count"] / (df_exploded["vote_count"] + m)) * df_exploded["vote_average"] +
    (m / (df_exploded["vote_count"] + m)) * C
).round(2)

end_time = time.time()
end_memory = psutil.Process(os.getpid()).memory_info().rss / (1024 * 1024)
end_cpu = psutil.Process(os.getpid()).cpu_percent(interval=None)

execution_time = end_time - start_time
memory_delta = end_memory - start_memory
col_memory = df_exploded["weighted_score"].memory_usage(deep=True) / 1024**2
rows_processed = len(df_exploded)

# Store metrics for summary
analysis4_time = execution_time
analysis4_memory = memory_delta

print(f"\n RESULTS:")
print(f"Global average rating (C): {C:.2f}")
print(f"75th percentile vote count (m): {m:.0f}")
print(f"Sample of weighted scores:")
sample_df = df_exploded[["title", "vote_average", "vote_count", "weighted_score"]].dropna().head(10)
print(sample_df)

print(f"\n BENCHMARKS:")
print(f"Time: {execution_time:.2f} seconds")
print(f"Memory delta: {memory_delta:.2f} MB")
print(f"Memory total: {end_memory:.2f} MB")
print(f"CPU: {end_cpu:.1f}%")
print(f"Rows processed: {rows_processed:,}")
print(f"Processing speed: {rows_processed/execution_time:,.0f} rows/second")
print(f"Weighted column memory cost: {col_memory:.2f} MB")


Q4: WEIGHTED POPULARITY SCORE

 RESULTS:
Global average rating (C): 3.34
75th percentile vote count (m): 6
Sample of weighted scores:
                 title  vote_average  vote_count  weighted_score
0                Ariel           7.1         367            7.04
0                Ariel           7.1         367            7.04
0                Ariel           7.1         367            7.04
0                Ariel           7.1         367            7.04
1  Shadows in Paradise           7.3         430            7.25
1  Shadows in Paradise           7.3         430            7.25
1  Shadows in Paradise           7.3         430            7.25
2           Four Rooms           5.9        2780            5.89
3       Judgment Night           6.5         360            6.45
3       Judgment Night           6.5         360            6.45

 BENCHMARKS:
Time: 0.02 seconds
Memory delta: 0.69 MB
Memory total: 3375.69 MB
CPU: 0.0%
Rows processed: 976,441
Processing speed: 59,952,723 rows/se

In [10]:
print("\n" + "="*80)
print("Q5: TOP PERFORMING GENRE PER YEAR")
print("="*80)

start_time = time.time()
start_memory = psutil.Process(os.getpid()).memory_info().rss / (1024 * 1024)
start_cpu = psutil.Process(os.getpid()).cpu_percent(interval=None)

# Rank and get top genre per year by average rating
ranked = (
    grouped.sort_values(["year", "vote_average"], ascending=[True, False])
           .groupby("year")
           .first()
           .reset_index()
)

# Add movie count for context
ranked = ranked.merge(
    volume, 
    on=["year", "genres"], 
    how="left"
)

end_time = time.time()
end_memory = psutil.Process(os.getpid()).memory_info().rss / (1024 * 1024)
end_cpu = psutil.Process(os.getpid()).cpu_percent(interval=None)

execution_time = end_time - start_time
memory_delta = end_memory - start_memory
years_covered = len(ranked)

# Store metrics for summary
analysis5_time = execution_time
analysis5_memory = memory_delta

print(f"\n RESULTS:")
print(f"Years with top genre data: {years_covered}")
print(f"Year range: {ranked['year'].min()} - {ranked['year'].max()}")
print("\nTop performing genre by year (sample):")
print(ranked.head(20))

print(f"\n BENCHMARKS:")
print(f"Time: {execution_time:.2f} seconds")
print(f"Memory delta: {memory_delta:.2f} MB")
print(f"Memory total: {end_memory:.2f} MB")
print(f"CPU: {end_cpu:.1f}%")
print(f"Years processed: {years_covered}")
print(f"Processing speed: {years_covered/execution_time:.1f} years/second")


Q5: TOP PERFORMING GENRE PER YEAR

 RESULTS:
Years with top genre data: 143
Year range: 1888 - 2069

Top performing genre by year (sample):
    year           genres  vote_average  movie_count
0   1888      Documentary          5.68            2
1   1890      Documentary          4.70            2
2   1891      Documentary          4.90            1
3   1892      Documentary          3.50            1
4   1893            Drama          4.15            2
5   1894            Music          6.10            1
6   1895          History          6.20            1
7   1896          Mystery          6.20            1
8   1897          Fantasy          6.30            1
9   1898        Animation          7.00            1
10  1899           Family          6.20            1
11  1900        Animation          6.37            1
12  1901           Horror          5.80            2
13  1902  Science Fiction          7.90            1
14  1903          Western          7.00            1
15  1904   

In [11]:
print("\n" + "="*80)
print("PERFORMANCE SUMMARY")
print("="*80)

# Collect results from all analyses
results = [
    {"Analysis": "Data Loading", "Time (s)": load_time if 'load_time' in locals() else 0, 
     "Memory (MB)": load_memory if 'load_memory' in locals() else 0},
    
    {"Analysis": "Data Preparation", "Time (s)": prep_time if 'prep_time' in locals() else 0, 
     "Memory (MB)": prep_memory if 'prep_memory' in locals() else 0},
    
    {"Analysis": "Q1: Explode Genres", "Time (s)": analysis1_time if 'analysis1_time' in locals() else 0, 
     "Memory (MB)": analysis1_memory if 'analysis1_memory' in locals() else 0},
    
    {"Analysis": "Q2: Yearly Avg Rating", "Time (s)": analysis2_time if 'analysis2_time' in locals() else 0, 
     "Memory (MB)": analysis2_memory if 'analysis2_memory' in locals() else 0},
    
    {"Analysis": "Q3: Genre Volume", "Time (s)": analysis3_time if 'analysis3_time' in locals() else 0, 
     "Memory (MB)": analysis3_memory if 'analysis3_memory' in locals() else 0},
    
    {"Analysis": "Q4: Weighted Score", "Time (s)": analysis4_time if 'analysis4_time' in locals() else 0, 
     "Memory (MB)": analysis4_memory if 'analysis4_memory' in locals() else 0},
    
    {"Analysis": "Q5: Top Genre/Year", "Time (s)": analysis5_time if 'analysis5_time' in locals() else 0, 
     "Memory (MB)": analysis5_memory if 'analysis5_memory' in locals() else 0}
]

# Create DataFrame for better display
summary_df = pd.DataFrame(results)

# Calculate totals
total_time = summary_df["Time (s)"].sum()
total_memory = summary_df["Memory (MB)"].sum()

print("\n" + "-"*70)
print(f"{'Analysis':<45} {'Time (s)':<12} {'Memory (MB)':<12}")
print("-"*70)

for _, row in summary_df.iterrows():
    print(f"{row['Analysis']:<45} {row['Time (s)']:<12.2f} {row['Memory (MB)']:<12.2f}")

print("-"*70)
print(f"{'TOTAL':<45} {total_time:<12.2f} {total_memory:<12.2f}")
print("-"*70)

# Calculate end time and memory
end_total = time.time()
end_memory_total = psutil.Process(os.getpid()).memory_info().rss / (1024 * 1024)
end_cpu_total = psutil.Process(os.getpid()).cpu_percent(interval=None)

total_execution_time = end_total - start_total
total_memory_delta = end_memory_total - start_memory_total


PERFORMANCE SUMMARY

----------------------------------------------------------------------
Analysis                                      Time (s)     Memory (MB) 
----------------------------------------------------------------------
Data Loading                                  9.13         2254.89     
Data Preparation                              0.40         129.52      
Q1: Explode Genres                            1.66         783.77      
Q2: Yearly Avg Rating                         0.06         31.88       
Q3: Genre Volume                              0.05         23.89       
Q4: Weighted Score                            0.02         0.69        
Q5: Top Genre/Year                            0.00         0.39        
----------------------------------------------------------------------
TOTAL                                         11.32        3225.02     
----------------------------------------------------------------------


#### Map Reduce Implementation for qeury 1 : Explode Genres 

In [28]:


from pyspark.sql import SparkSession
import time, psutil, os, ast
import pandas as pd

try:
    spark
except NameError:
    spark = SparkSession.builder \
        .appName("GenrePopularityQ1_MapReduce") \
        .master("local[*]") \
        .getOrCreate()

sc = spark.sparkContext

print("\n" + "="*80)
print("MAPREDUCE Q1: EXPLODE MULTI-GENRE STRINGS")
print("="*80)

start_time = time.time()
start_memory = psutil.Process(os.getpid()).memory_info().rss / (1024 * 1024)

# Prepare data
movies_q1 = movies[["id", "title", "genres", "year", "vote_average", "vote_count"]].copy()
records = movies_q1.to_dict("records")
movies_rdd = sc.parallelize(records)

# Helper functions
def safe_parse_genres(genres):
    if isinstance(genres, list):
        return genres
    if isinstance(genres, str):
        try:
            if genres.startswith("["):
                parsed = ast.literal_eval(genres)
                if isinstance(parsed, list):
                    return parsed
            return [g.strip() for g in genres.split(",") if g.strip()]
        except:
            return ["Unknown"]
    return ["Unknown"]

def explode_genres(record):
    genres_list = safe_parse_genres(record.get("genres"))
    return [
        {
            "id": record.get("id"),
            "title": record.get("title"),
            "year": record.get("year"),
            "vote_average": record.get("vote_average"),
            "vote_count": record.get("vote_count"),
            "genres": genre
        }
        for genre in genres_list
    ]

# MapReduce (flatMap = Map step)
exploded_rdd = movies_rdd.flatMap(explode_genres)

# Collecting the result
df_exploded_mr = pd.DataFrame(exploded_rdd.collect())

end_time = time.time()
end_memory = psutil.Process(os.getpid()).memory_info().rss / (1024 * 1024)

# Results
print(f"\nRows before: {len(movies_q1):,}")
print(f"Rows after: {len(df_exploded_mr):,}")
print(f"Expansion factor: {len(df_exploded_mr)/len(movies_q1):.2f}x")

print(f"\nTime: {end_time - start_time:.2f}s")
print(f"Memory delta: {end_memory - start_memory:.2f} MB")

df_exploded_mr.head()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/03/19 11:18:20 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable



MAPREDUCE Q1: EXPLODE MULTI-GENRE STRINGS


26/03/19 11:18:24 WARN TaskSetManager: Stage 0 contains a task of very large size (4349 KiB). The maximum recommended task size is 1000 KiB.



Rows before: 590,202
Rows after: 976,441
Expansion factor: 1.65x

Time: 3.49s
Memory delta: 476.62 MB


,id,title,year,vote_average,vote_count,genres
0,2,Ariel,1988,7.1,367,Comedy
1,2,Ariel,1988,7.1,367,Drama
2,2,Ariel,1988,7.1,367,Romance
3,2,Ariel,1988,7.1,367,Crime
4,3,Shadows in Paradise,1986,7.3,430,Comedy
